## Is the market equally efficient everywhere, or does it break down under specific conditions?

Every notebook so far reports a single pooled result: no overreaction, no underreaction, reasonable calibration on average. But "efficient on average" can hide real variation. This notebook re-cuts `calibration_analysis.ipynb`'s Brier-score methodology across three conditions where efficiency might plausibly differ:

1. **Tournament progression** &mdash; did pricing get sharper over the tournament as the market matured?
2. **Match-level attention** (total `volume`, from `attention_analysis.ipynb`) &mdash; are the most-watched matches priced *better* (more sophisticated money) or *worse* (more noise-driven retail money chasing popular games)?
3. **Checkpoint-level liquidity** (`open_interest` at that specific point in time) &mdash; does pricing get worse specifically when positions are thin?

"Efficient on average, breaks down under X" is a much sharper claim than a single pooled coefficient &mdash; this is that test.

### A data-quality fix worth calling out

Some Kalshi candles have zero volume and report a null `price_close` (no trades that minute, no close price to report) &mdash; 221 of 74,754 candle-minutes. `calibration_analysis.ipynb`'s checkpoint lookup ("last candle at or before this time") could land on exactly one of these, silently including a `NaN` price in a handful of observations. Pandas' `.mean()` quietly skips `NaN`s, so the numbers that notebook reported were unaffected (verified directly: only 29 of 2496 checkpoint lookups were hit, and every downstream aggregation there happens to skip NaNs safely) &mdash; but `scipy.stats.ttest_ind` does **not** skip `NaN`s by default, and this notebook's group-comparison tests hit that immediately as a hard `nan` result.

The right fix isn't dropping those rows &mdash; a zero-volume candle *is* a real, informative thin-liquidity moment, and dropping it would bias the liquidity test in exactly the direction that matters here. Instead, each market's price series is forward-filled (last traded price carries forward through zero-volume gaps) before any checkpoint is extracted.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

# Notebook lives in code/, so data/ and output/ live one level up.
ROOT = Path.cwd().parent
FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = [180, 150, 120, 90, 60, 30, 15, 5]

In [ ]:
markets = pd.read_parquet(ROOT / "data/kalshi/kxwcgame_markets.parquet")
candles = pd.read_parquet(ROOT / "data/kalshi/candlesticks/kxwcgame_minute.parquet")

markets = markets[markets["status"] == "finalized"].copy()
markets["close_time"] = pd.to_datetime(markets["close_time"])
markets["outcome"] = (markets["result"] == "yes").astype(int)

match_volume = markets.groupby("event_ticker")["volume"].sum().rename("match_volume")
markets = markets.merge(match_volume, on="event_ticker")

median_close = markets["close_time"].median()
markets["tournament_half"] = np.where(markets["close_time"] <= median_close, "early", "late")

vol_by_match = markets.drop_duplicates("event_ticker")[["event_ticker", "match_volume"]]
vol_by_match["volume_tercile"] = pd.qcut(vol_by_match["match_volume"], 3, labels=["low", "mid", "high"])
markets = markets.merge(vol_by_match[["event_ticker", "volume_tercile"]], on="event_ticker")

print(f"{len(markets)} finalized markets; tournament split at {median_close.date()}")
print(markets.drop_duplicates("event_ticker")["volume_tercile"].value_counts())

In [ ]:
candles = candles.merge(markets[["market_ticker", "close_time"]], on="market_ticker", how="inner")


def prep(g: pd.DataFrame) -> pd.DataFrame:
    g = g.sort_values("timestamp").copy()
    g["price_close"] = g["price_close"].ffill()  # carry last traded price through zero-volume gaps
    return g


candles_by_ticker = {t: prep(g) for t, g in candles.groupby("market_ticker")}

rows = []
for m in markets.itertuples():
    grp = candles_by_ticker.get(m.market_ticker)
    if grp is None or grp.empty:
        continue
    for cp in CHECKPOINTS:
        target = m.close_time - pd.Timedelta(minutes=cp)
        before = grp[grp["timestamp"] <= target]
        if before.empty:
            continue
        row = before.iloc[-1]
        rows.append(
            {
                "market_ticker": m.market_ticker, "checkpoint": cp,
                "price": row["price_close"], "open_interest": row["open_interest"],
                "outcome": m.outcome, "tournament_half": m.tournament_half,
                "volume_tercile": m.volume_tercile, "match_volume": m.match_volume,
            }
        )

obs = pd.DataFrame(rows)
obs["sq_err"] = (obs["price"] - obs["outcome"]) ** 2
obs["oi_median"] = obs.groupby("checkpoint")["open_interest"].transform("median")
obs["liquidity"] = np.where(obs["open_interest"] <= obs["oi_median"], "thin", "thick")
print(f"{len(obs)} price/outcome observations, {obs['price'].isna().sum()} still NaN after forward-fill")

### Three univariate cuts

In [ ]:
print("--- Brier score by tournament half ---")
print(obs.groupby("tournament_half")["sq_err"].agg(["mean", "count"]))
_, p_half = stats.ttest_ind(
    obs.loc[obs["tournament_half"] == "early", "sq_err"], obs.loc[obs["tournament_half"] == "late", "sq_err"]
)
print(f"early vs. late, t-test p = {p_half:.3f}\n")

print("--- Brier score by match-level attention (volume tercile) ---")
print(obs.groupby("volume_tercile", observed=True)["sq_err"].agg(["mean", "count"]))
_, p_vol = stats.ttest_ind(
    obs.loc[obs["volume_tercile"] == "low", "sq_err"], obs.loc[obs["volume_tercile"] == "high", "sq_err"]
)
print(f"low vs. high, t-test p = {p_vol:.2e}\n")

print("--- Brier score by checkpoint-level liquidity (open interest, within-checkpoint median split) ---")
print(obs.groupby("liquidity")["sq_err"].agg(["mean", "count"]))
_, p_liq = stats.ttest_ind(obs.loc[obs["liquidity"] == "thin", "sq_err"], obs.loc[obs["liquidity"] == "thick", "sq_err"])
print(f"thin vs. thick, t-test p = {p_liq:.3f}")

### The two real effects overlap — untangling them

High-volume matches are almost always "thick" liquidity and low-volume matches almost always "thin" (checked directly: a simple crosstab shows the two splits agree far more often than chance) &mdash; match-level attention and checkpoint-level open interest aren't independent, so the two univariate tests above aren't two separate confirmations. A single regression with both predictors *and* checkpoint fixed effects (to remove the mechanical fact that open interest and precision both trend with time-to-close) tests whether each has an independent effect on the other's own terms.

In [ ]:
obs["log_volume_z"] = (np.log(obs["match_volume"]) - np.log(obs["match_volume"]).mean()) / np.log(obs["match_volume"]).std()
obs["log_oi_z"] = (np.log(obs["open_interest"]) - np.log(obs["open_interest"]).mean()) / np.log(obs["open_interest"]).std()
obs["checkpoint_cat"] = obs["checkpoint"].astype(str)

model = smf.ols("sq_err ~ log_volume_z + log_oi_z + C(checkpoint_cat)", data=obs).fit()
print(model.summary())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

ax = axes[0]
for half, color in [("early", "tab:blue"), ("late", "tab:orange")]:
    sub = obs[obs["tournament_half"] == half].groupby("checkpoint")["sq_err"].mean().sort_index(ascending=False)
    ax.plot(sub.index, sub.values, marker="o", color=color, label=half)
ax.invert_xaxis()
ax.set_xlabel("minutes before close")
ax.set_ylabel("Brier score")
ax.set_title(f"By tournament progression\n(p = {p_half:.2f}, not significant)")
ax.legend(fontsize=8)

ax = axes[1]
for tercile, color in [("low", "tab:green"), ("mid", "tab:orange"), ("high", "tab:red")]:
    sub = obs[obs["volume_tercile"] == tercile].groupby("checkpoint")["sq_err"].mean().sort_index(ascending=False)
    ax.plot(sub.index, sub.values, marker="o", color=color, label=f"{tercile} volume")
ax.invert_xaxis()
ax.set_xlabel("minutes before close")
ax.set_title(f"By match-level attention\n(low vs. high, p = {p_vol:.1e})")
ax.legend(fontsize=8)

ax = axes[2]
ci = model.conf_int()
for predictor, label, y in [("log_volume_z", "match volume\n(attention)", 1), ("log_oi_z", "open interest\n(liquidity)", 0)]:
    coef = model.params[predictor]
    lo, hi = ci.loc[predictor]
    ax.errorbar([coef], [y], xerr=[[coef - lo], [hi - coef]], fmt="o", capsize=4, color="black")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_yticks([0, 1])
ax.set_yticklabels(["open interest\n(liquidity)", "match volume\n(attention)"])
ax.set_xlabel("effect on squared error (95% CI)\ncontrolling for the other + checkpoint")
ax.set_title("Independent effects\n(both p < 0.001)")

#fig.suptitle("Where does calibration break down? Attention hurts it, accumulated positions help it")
fig.tight_layout()
out_path = FIG_DIR / "heterogeneity_analysis.png"
fig.savefig(out_path, dpi=150)
print(f"Saved {out_path}")

### Robustness check: is the attention effect really just group stage vs. knockout?

`attention_analysis.ipynb` found group-stage matches draw far more volume than knockout matches. That raises an obvious question: is "high volume predicts worse calibration" actually just a disguised version of "group stage is priced differently than knockouts," rather than attention itself doing anything? Adding tournament stage as a control (both a simple group-vs-knockout flag and the full 7-level stage) answers this directly.

In [ ]:
TEAM_NAME_ALIASES = {
    "Cape Verde": "Cabo Verde", "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Congo DR": "DR Congo", "Ivory Coast": "Côte d'Ivoire", "IR Iran": "Iran",
    "Turkiye": "Türkiye", "Curacao": "Curaçao", "Korea Republic": "South Korea",
}

sofa_schedule = pd.read_parquet(ROOT / "data/sofascore/schedule.parquet")
sofa_schedule["stage"] = sofa_schedule["round_name"].fillna("Group stage")
sofa_schedule["team_set"] = [frozenset(t) for t in zip(sofa_schedule["home_team"], sofa_schedule["away_team"])]

parts = markets["event_title"].str.split(" vs ", n=1, expand=True)
markets["team1_kalshi"] = parts[0].str.strip()
markets["team2_kalshi"] = parts[1].str.split(":").str[0].str.strip()
markets["team_set"] = [
    frozenset(t)
    for t in zip(markets["team1_kalshi"].replace(TEAM_NAME_ALIASES), markets["team2_kalshi"].replace(TEAM_NAME_ALIASES))
]

markets_with_stage = markets.merge(sofa_schedule[["team_set", "stage"]].drop_duplicates(), on="team_set", how="left")
print(f"unmatched to stage: {markets_with_stage['stage'].isna().sum()} / {len(markets_with_stage)}")

stage_lookup = markets_with_stage.set_index("market_ticker")["stage"]
obs["stage"] = obs["market_ticker"].map(stage_lookup)
obs["is_knockout"] = obs["stage"] != "Group stage"

print("\nmean match volume by stage (confirms the attention_analysis.ipynb pattern):")
print(obs.drop_duplicates("market_ticker").groupby("stage")["match_volume"].mean().sort_values(ascending=False))

In [ ]:
model_no_stage = smf.ols("sq_err ~ log_volume_z + log_oi_z + C(checkpoint_cat)", data=obs).fit()
model_binary_stage = smf.ols("sq_err ~ log_volume_z + log_oi_z + C(checkpoint_cat) + is_knockout", data=obs).fit()
model_full_stage = smf.ols("sq_err ~ log_volume_z + log_oi_z + C(checkpoint_cat) + C(stage)", data=obs).fit()

comparison = pd.DataFrame(
    {
        "no stage control": [model_no_stage.params["log_volume_z"], model_no_stage.pvalues["log_volume_z"]],
        "+ group vs. knockout": [model_binary_stage.params["log_volume_z"], model_binary_stage.pvalues["log_volume_z"]],
        "+ full 7-stage control": [model_full_stage.params["log_volume_z"], model_full_stage.pvalues["log_volume_z"]],
    },
    index=["log_volume_z coefficient", "p-value"],
)
print(comparison)
print(f"\nis_knockout effect on its own: coef = {model_binary_stage.params['is_knockout[T.True]']:+.4f}, p = {model_binary_stage.pvalues['is_knockout[T.True]']:.2e}")
print(f"(positive = knockout matches are worse-calibrated than group stage, controlling for volume and open interest)")

### Results so far

**This is the strongest, most robust finding in the whole project so far — and it survives the obvious follow-up challenge.**

**Tournament progression: no effect.** Early- and late-tournament Brier scores are essentially identical at every checkpoint (0.128 vs. 0.131, p = 0.76) — no sign the market got sharper (or noisier) as the tournament wore on.

**Match-level attention: a large, highly significant, and remarkably consistent effect.** High-volume matches are worse-calibrated than low-volume matches at *every single checkpoint*, not just on average (visible directly in the middle panel — the three lines barely cross at all across the full 3-hours-to-5-minutes range). Pooled: low-volume Brier = 0.103, high-volume Brier = 0.151, p = 2.6×10⁻⁷. This is exactly the kind of attention-driven mispricing `attention_analysis.ipynb` went looking for and didn't find at the goal-event level — it shows up here instead, at the whole-match level.

**Checkpoint-level liquidity: a real, independent, opposite-signed effect.** Thin-open-interest moments are worse-calibrated than thick ones (0.139 vs. 0.120, p = 0.018) — but match volume and open interest are heavily confounded (high-volume matches are almost always "thick"), so this isn't a second independent confirmation on its own. Untangling the two with a joint regression (both predictors + checkpoint fixed effects) shows they survive together: **higher match volume independently predicts *worse* calibration (p < 0.001), while higher open interest independently predicts *better* calibration (p < 0.001)**, each controlling for the other and for time-to-close.

**Robustness check: is the attention effect just a disguised group-stage-vs-knockout effect? No — it gets *stronger*, not weaker.** `attention_analysis.ipynb` already showed group-stage matches draw much more volume, which raised a real possibility that "attention" was just proxying for tournament stage. Adding a stage control settles it: the `log_volume_z` coefficient goes from **0.047 (no stage control) to 0.072 (group vs. knockout) to 0.075 (full 7-level stage)**, with the p-value tightening from 1.4×10⁻²³ to roughly 1.7×10⁻³⁹. Controlling for stage doesn't explain the volume effect away — it sharpens it, because group-stage matches (high volume) turn out to be independently *better*-calibrated for reasons unrelated to volume, which was partly masking the pure attention effect in the uncontrolled model.

That control also surfaced a genuine second finding along the way: **knockout matches are independently worse-calibrated than group-stage matches**, controlling for both volume and open interest (coefficient +0.093, p = 1.3×10⁻¹⁷). Plausible story: sudden-death stakes, extra time/penalties, and more emotionally-charged betting on elimination games — but that's this notebook's next open thread, not something tested here.

**The honest interpretation:** two different kinds of "more money" pull in opposite directions — a lot of trading *flow* (volume, turnover) in a match is associated with worse pricing, while a lot of accumulated *positions* (open interest) is associated with better pricing, and both hold up after accounting for tournament stage. Read generously, that's consistent with a noise-trader story: popular matches draw more casual, less-informed churn, while open interest reflects a market that's actually converged on settled views. That interpretation is plausible, not proven — this is a correlational finding, and the mechanism isn't tested directly here.

**Caveats:**

- Same non-independence caveat as `calibration_analysis.ipynb`: the 3 markets per match aren't independent draws, so these p-values are a bit optimistic. Given how large and consistent the volume effect is (visible at every single checkpoint, not just in aggregate, and *strengthening* under a stage control), it would very likely survive a proper match-clustered correction — but that correction hasn't been run.
- "Attention" and "liquidity" here are still just two Kalshi-native numbers (`volume`, `open_interest`), not a direct measure of *who* is trading or *why* — the noise-trader interpretation is a reasonable story that fits the pattern, not a demonstrated mechanism.
- The knockout-stage effect uncovered here is new and untested beyond this one regression — worth its own look before treating it as more than a suggestive finding.